***I HAVE USED FUNCTIONS TO FETCH DATA OF BOTH TICKERS FOR ALL STRATEGIES***

In [98]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import yfinance as yf

# **PART 1**


In [99]:
adani= yf.download ("ADANIGREEN.NS",start='2024-05-22',end='2025-05-23',interval="1d")
bel= yf.download ("BEL.NS",start='2024-05-22',end='2025-05-23',interval="1d")
adani.columns= adani.columns.droplevel(1)
bel.columns= bel.columns.droplevel(1)
adani=adani.reset_index()
bel=bel.reset_index()
bel

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


Price,Date,Close,High,Low,Open,Volume
0,2024-05-22,281.276062,282.168681,262.134197,274.730139,85801218
1,2024-05-23,286.135895,290.896549,280.879338,283.953911,70631009
2,2024-05-24,294.764587,297.442444,283.656354,285.639965,63609206
3,2024-05-27,292.037170,300.715469,287.276485,299.277339,49890427
4,2024-05-28,286.730988,293.326488,282.714157,293.276910,33893805
...,...,...,...,...,...,...
244,2025-05-16,363.899994,371.149994,349.600006,351.600006,68855818
245,2025-05-19,363.750000,373.500000,362.549988,372.899994,52003735
246,2025-05-20,363.799988,371.000000,358.500000,369.399994,67088831
247,2025-05-21,383.000000,383.899994,364.000000,366.899994,88688170


**STRATEGY 1 FOR BOTH**

In [100]:
def strategy1(data):
  EMA_14= data["Close"].ewm(span=14).mean()
  i=1
  data["signal1"]=0
  while i<len(data):
    if (data.loc[i,"Close"]>EMA_14[i]) and (data.loc[i-1,"Close"]<EMA_14[i-1]):
      data.loc[i,"signal1"]="buy"
    elif (data.loc[i,"Close"]<EMA_14[i]) and (data.loc[i-1,"Close"]>EMA_14[i-1]):
      data.loc[i,"signal1"]="sell"
    i+=1
  i=0
  while data.loc[i,"signal1"]!="buy":  ### removing all sells before first buy..
    data.loc[i,"signal1"]=0
    i+=1
  return data
strategy1(bel)

<ipython-input-100-4842ff817d36>:9: FutureWarning:

Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'sell' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.



Price,Date,Close,High,Low,Open,Volume,signal1
0,2024-05-22,281.276062,282.168681,262.134197,274.730139,85801218,0
1,2024-05-23,286.135895,290.896549,280.879338,283.953911,70631009,0
2,2024-05-24,294.764587,297.442444,283.656354,285.639965,63609206,0
3,2024-05-27,292.037170,300.715469,287.276485,299.277339,49890427,0
4,2024-05-28,286.730988,293.326488,282.714157,293.276910,33893805,0
...,...,...,...,...,...,...,...
244,2025-05-16,363.899994,371.149994,349.600006,351.600006,68855818,0
245,2025-05-19,363.750000,373.500000,362.549988,372.899994,52003735,0
246,2025-05-20,363.799988,371.000000,358.500000,369.399994,67088831,0
247,2025-05-21,383.000000,383.899994,364.000000,366.899994,88688170,0


**STRATEGY 2 FOR BOTH**

In [101]:
def strategy2(data):
    EMA_14= data["Close"].ewm(span=14).mean()
    buy_band= EMA_14 + (data["High"]-data["Low"])
    sell_band= EMA_14 - (data["High"]-data["Low"])
    i=1
    data["signal2"]=0
    while i<len(data):
      if data.loc[i,"Close"]>buy_band[i] and data.loc[i-1,"Close"]<=buy_band[i-1]:
        data.loc[i,"signal2"]="buy"
      elif data.loc[i,"Close"]<sell_band[i] and data.loc[i-1,"Close"]>=sell_band[i-1]:
        data.loc[i,"signal2"]="sell"
      i+=1

    i=0
    while data.loc[i,"signal2"]!="buy":  ### removing all sells before first buy..
      data.loc[i,"signal2"]=0
      i+=1

    i=0
    while i<len(data)-1:            ### Keeping one trade active at a time....removing repetitive buy and sell....
      if data.loc[i,"signal2"]=="sell":
        j=i+1
        while j<len(data) and data.loc[j,"signal2"]!="buy":
          data.loc[j,"signal2"]=0
          j+=1
      i+=1
    i=0
    while i<len(data)-1:
      if data.loc[i,"signal2"]=="buy":
        j=i+1
        while j<len(data) and data.loc[j,"signal2"]!="sell":
          data.loc[j,"signal2"]=0
          j+=1
      i+=1
    return data
strategy2(bel)

<ipython-input-101-b0122baeb385>:9: FutureWarning:

Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'buy' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.



Price,Date,Close,High,Low,Open,Volume,signal1,signal2
0,2024-05-22,281.276062,282.168681,262.134197,274.730139,85801218,0,0
1,2024-05-23,286.135895,290.896549,280.879338,283.953911,70631009,0,0
2,2024-05-24,294.764587,297.442444,283.656354,285.639965,63609206,0,0
3,2024-05-27,292.037170,300.715469,287.276485,299.277339,49890427,0,0
4,2024-05-28,286.730988,293.326488,282.714157,293.276910,33893805,0,0
...,...,...,...,...,...,...,...,...
244,2025-05-16,363.899994,371.149994,349.600006,351.600006,68855818,0,0
245,2025-05-19,363.750000,373.500000,362.549988,372.899994,52003735,0,0
246,2025-05-20,363.799988,371.000000,358.500000,369.399994,67088831,0,0
247,2025-05-21,383.000000,383.899994,364.000000,366.899994,88688170,0,0


**STRATEGY 3 FOR BOTH**

In [102]:
def strategy3(data):
    EMA_14= data["Close"].ewm(span=14).mean()
    tr1= data["High"]-data["Low"]
    tr2= abs(data["High"]-data["Close"].shift(1))
    tr3= abs(data["Low"]-data["Close"].shift(1))
    tr= np.maximum(tr1,tr2,tr3)
    tr
    atr= tr.ewm(span=14).mean()
    buy_band1= EMA_14 + 1.5*atr
    sell_band1= EMA_14 - 1.5*atr
    i=1
    data["signal3"]=0
    while i<len(data):
      if data.loc[i,"Close"]>buy_band1[i] and data.loc[i-1,"Close"]<=buy_band1[i-1]:
        data.loc[i,"signal3"]="buy"
      elif data.loc[i,"Close"]<sell_band1[i] and data.loc[i-1,"Close"]>=sell_band1[i-1]:
        data.loc[i,"signal3"]="sell"
      i+=1

    i=0
    while data.loc[i,"signal3"]!="buy":  ### removing all sells before first buy..
      data.loc[i,"signal3"]=0
      i+=1
    i=0
    while i<len(data)-1:            ### Keeping one trade active at a time....removing repetitive buy and sell....
      if data.loc[i,"signal3"]=="sell":
        j=i+1
        while j<len(data) and data.loc[j,"signal3"]!="buy":
          data.loc[j,"signal3"]=0
          j+=1
      i+=1
    i=0
    while i<len(data)-1:
      if data.loc[i,"signal3"]=="buy":
        j=i+1
        while j<len(data) and data.loc[j,"signal3"]!="sell":
          data.loc[j,"signal3"]=0
          j+=1
      i+=1
    return data
strategy3(bel)

<ipython-input-102-32e8245a3acc>:15: FutureWarning:

Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'buy' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.



Price,Date,Close,High,Low,Open,Volume,signal1,signal2,signal3
0,2024-05-22,281.276062,282.168681,262.134197,274.730139,85801218,0,0,0
1,2024-05-23,286.135895,290.896549,280.879338,283.953911,70631009,0,0,0
2,2024-05-24,294.764587,297.442444,283.656354,285.639965,63609206,0,0,0
3,2024-05-27,292.037170,300.715469,287.276485,299.277339,49890427,0,0,0
4,2024-05-28,286.730988,293.326488,282.714157,293.276910,33893805,0,0,0
...,...,...,...,...,...,...,...,...,...
244,2025-05-16,363.899994,371.149994,349.600006,351.600006,68855818,0,0,0
245,2025-05-19,363.750000,373.500000,362.549988,372.899994,52003735,0,0,0
246,2025-05-20,363.799988,371.000000,358.500000,369.399994,67088831,0,0,0
247,2025-05-21,383.000000,383.899994,364.000000,366.899994,88688170,0,0,0


**TRADEWISE DATAFRAME** **(signal1=strategy1,...continue)**

In [103]:
def tradewise(data,signal):
    trade_index= []
    i=0
    while i<len(data):
       if data[signal].iloc[i]== "buy" or data[signal].iloc[i]== "sell":
         trade_index.append(i)
       i+=1
    col= [
        'Entry Index', 'Exit Index', 'Entry Date', 'Exit Date', 'Trade Duration',
        'Returns for the trade in percent', 'Type of trade', 'Max Drawdown for the trade',
        'Max dip for the trade'
    ]

    df = pd.DataFrame(columns=col)

    i=1
    while i<len(trade_index)-1:
        x = data['Close'].iloc[trade_index[i]]
        y = data['Close'].iloc[trade_index[i-1]]
        df.loc[i-1, 'Entry Index'] = trade_index[i-1]
        df.loc[i-1, 'Exit Index'] = trade_index[i]
        df.loc[i-1, 'Trade Duration'] = trade_index[i] - trade_index[i-1]
        df.loc[i-1,'Entry Date']=data['Date'].iloc[trade_index[i-1]]
        df.loc[i-1,'Exit Date']=data['Date'].iloc[trade_index[i]]
        df.loc[i-1, 'Returns for the trade in percent'] = ((x - y) * 100) / y

        if data[signal].iloc[trade_index[i-1]]== "buy":
            df.loc[i-1,'Type of trade']='Long'
        else:
            df.loc[i-1,'Type of trade']='Short'

        min=y
        j=trade_index[i-1] + 1
        while j< trade_index[i] + 1:
            if data['Close'].iloc[j] <min:
                min = data['Close'].iloc[j]
            j+=1
        df.loc[i-1, 'Max dip for the trade'] = ((min-y)/y)*100

        max1=data['Close'].iloc[trade_index[i-1]]
        max2=0
        j=trade_index[i-1]
        while j<trade_index[i]+1:
            if data['Close'].iloc[j] >max1:
                max1 = data['Close'].iloc[j]
            t=(max1-data['Close'].iloc[j])*100 /max1
            if t>max2:
                max2=t
            j+=1
        df.loc[i-1, 'Max Drawdown for the trade']= max2
        i+=1

    df = df.reset_index(drop=True)
    return df
tradewise(bel,"signal2")

,Entry Index,Exit Index,Entry Date,Exit Date,Trade Duration,Returns for the trade in percent,Type of trade,Max Drawdown for the trade,Max dip for the trade
0,8,40,2024-06-03 00:00:00,2024-07-19 00:00:00,32,-3.87573,Long,19.802288,-19.802288
1,40,65,2024-07-19 00:00:00,2024-08-26 00:00:00,25,0.401825,Short,10.611497,-6.219392
2,65,74,2024-08-26 00:00:00,2024-09-06 00:00:00,9,-7.531794,Long,7.531794,-7.531794
3,74,100,2024-09-06 00:00:00,2024-10-15 00:00:00,26,1.851205,Short,8.894189,-5.7299
4,100,106,2024-10-15 00:00:00,2024-10-23 00:00:00,6,-6.993262,Long,6.993262,-6.993262
5,106,113,2024-10-23 00:00:00,2024-11-01 00:00:00,7,7.444639,Short,1.264952,0.0
6,113,125,2024-11-01 00:00:00,2024-11-21 00:00:00,12,-4.57301,Long,8.746062,-4.57301
7,125,128,2024-11-21 00:00:00,2024-11-26 00:00:00,3,8.150292,Short,0,0.0
8,128,145,2024-11-26 00:00:00,2024-12-19 00:00:00,17,0.201415,Long,5.597723,0.0
9,145,175,2024-12-19 00:00:00,2025-01-31 00:00:00,30,-1.959799,Short,13.484093,-13.484093


**CAPITAL CALCULATION**

In [104]:
def capital(data,signal):
    trade_index= []
    i=0
    while i<len(data):
       if data[signal].iloc[i] == "buy" or data[signal].iloc[i] == "sell":
         trade_index.append(i)
       i+=1
    cap= [100000]
    j=0
    amt=100000
    quantity=0
    i=0
    while i<len(data)-1:
        if j<len(trade_index) and i==trade_index[j]:
            if data[signal].iloc[i]== "buy":
                quantity= int(amt/data['Close'].iloc[i])
                amt -= quantity*data['Close'].iloc[i]
            elif data[signal].iloc[i]== "sell":
                amt += quantity*data['Close'].iloc[i]
                quantity =0
            cap.append(amt)
            j+=1
        else:
            cap.append(amt + quantity*data['Close'].iloc[i])
        i+=1
    data['Capital']=cap
    return data
capital(bel,"signal2")

Price,Date,Close,High,Low,Open,Volume,signal1,signal2,signal3,Capital
0,2024-05-22,281.276062,282.168681,262.134197,274.730139,85801218,0,0,0,100000.000000
1,2024-05-23,286.135895,290.896549,280.879338,283.953911,70631009,0,0,0,100000.000000
2,2024-05-24,294.764587,297.442444,283.656354,285.639965,63609206,0,0,0,100000.000000
3,2024-05-27,292.037170,300.715469,287.276485,299.277339,49890427,0,0,0,100000.000000
4,2024-05-28,286.730988,293.326488,282.714157,293.276910,33893805,0,0,0,100000.000000
...,...,...,...,...,...,...,...,...,...,...
244,2025-05-16,363.899994,371.149994,349.600006,351.600006,68855818,0,0,0,91720.672119
245,2025-05-19,363.750000,373.500000,362.549988,372.899994,52003735,0,0,0,95244.172119
246,2025-05-20,363.799988,371.000000,358.500000,369.399994,67088831,0,0,0,95205.023712
247,2025-05-21,383.000000,383.899994,364.000000,366.899994,88688170,0,0,0,95218.070526


**DAILY DATAFRAME**

In [105]:
def daily_df(data,signal):
  col=['Portfolio value','Quantity of Stocks','Profit from initial amount(%)']
  df1=pd.DataFrame(columns=col)
  i=0
  while i<len(data):
      df1.loc[i,'Portfolio value']=data["Capital"].iloc[i]
      df1.loc[i,'Profit from initial amount(%)']=((data["Capital"].iloc[i]-100000)*100)/100000
      if data[signal].iloc[i]=="buy" and i>=1:
          df1.loc[i,'Quantity of Stocks']=df1.loc[i-1,'Quantity of Stocks']+int(data["Capital"].iloc[i]/data['Close'].iloc[i])
      elif data[signal].iloc[i]=="sell" and i >=1:
          df1.loc[i,'Quantity of Stocks']=df1.loc[i-1,'Quantity of Stocks']-int(data["Capital"].iloc[i]/data['Close'].iloc[i])
      elif data[signal].iloc[i]==0 and i >=1:
          df1.loc[i,'Quantity of Stocks']=df1.loc[i-1,'Quantity of Stocks']
      else:
          if data[signal].iloc[i]=="buy":
              df1.loc[i,'Quantity of Stocks']+=int(data["Capital"].iloc[i]/data['Close'].iloc[i])
          elif data[signal].iloc[i]=="sell":
              df1.loc[i,'Quantity of Stocks']-=int(data["Capital"].iloc[i]/data['Close'].iloc[i])
          else:
              df1.loc[i,'Quantity of Stocks']=0
      i+=1
  return df1
daily_df(bel,"signal2")

,Portfolio value,Quantity of Stocks,Profit from initial amount(%)
0,100000.0,0,0.0
1,100000.0,0,0.0
2,100000.0,0,0.0
3,100000.0,0,0.0
4,100000.0,0,0.0
...,...,...,...
244,91720.672119,227,-8.279328
245,95244.172119,227,-4.755828
246,95205.023712,227,-4.794976
247,95218.070526,227,-4.781929


In [106]:
fig=go.Figure()
fig.add_trace(go.Scatter(x=bel["Date"], y=daily_df(bel,"signal2")["Portfolio value"], line = dict(color='blue', width=1)))
fig.update_layout(title=dict(text="Portfolio value over time",x=0.45,y=0.9,font=dict(size=30)),yaxis_title="portfolio value",xaxis_title="Date")
fig.show()

**METRICS**

In [107]:
def metrics(data,signal):
  amt = 100000
  quant = int(amt / data['Close'].iloc[0])
  z = quant * data['Close'].iloc[-1]
  r = z - amt
  rp = ((z - amt) * 100) / amt
  ct = 0
  i=0
  while i<len(data):
    if data[signal].iloc[i] =="buy":
        ct += 1
    i+=1
  print("Benchmark return =", r, "\nas a percentage=", rp, "%")
  print("\nNumber of closed trades = ", ct)

  ct1 = 0
  ct2 = 0
  i=0
  while i<len(tradewise(bel,"signal2")):
      if tradewise(bel,"signal2")['Returns for the trade in percent'].iloc[i] < 0:
          ct1 += 1
      else:
          ct2 += 1
      i+=1
  print("Number of Winning trades = ", ct2, "\nNumber of Losing trades = ", ct1)

  mean_time=tradewise(bel,"signal2")['Trade Duration'].mean()
  max_time=tradewise(bel,"signal2")['Trade Duration'].max()
  print("\nMaximum holding time = ",max_time,"\nMean holding time = ",(int)(mean_time))

  g_profit=data['Capital'].iloc[len(data)-1]-data['Capital'].iloc[0]
  n_profit=g_profit-(ct*20)
  print("\nGross Profit = ",g_profit,"\nNet Profit = ",n_profit)

  mean_drawdown=tradewise(bel,"signal2")['Max Drawdown for the trade'].mean()
  mean_dip=tradewise(bel,"signal2")['Max dip for the trade'].mean()
  max_dip=tradewise(bel,"signal2")['Max dip for the trade'].min()
  max_drawdown=tradewise(bel,"signal2")['Max Drawdown for the trade'].max()
  print("\nAverage drawdown for the trade = ",mean_drawdown,"%\n Maximum drawdown for the trade = ",max_drawdown,"%")
  print("\nAverage dip for the trade = ",mean_dip,"%\n Maximum dip for the trade = ",max_dip,"%")

  std_returns=tradewise(bel,"signal2")['Returns for the trade in percent'].std()
  mean_returns=tradewise(bel,"signal2")['Returns for the trade in percent'].mean()
  sharpe=(mean_returns/std_returns)
  print("Sharpe_like Ratio = ", sharpe)
metrics(bel,"signal2")

Benchmark return = 36106.99783325195 
as a percentage= 36.10699783325195 %

Number of closed trades =  7
Number of Winning trades =  5 
Number of Losing trades =  6

Maximum holding time =  32 
Mean holding time =  15

Gross Profit =  229.27371215820312 
Net Profit =  89.27371215820312

Average drawdown for the trade =  8.396084187871786 %
 Maximum drawdown for the trade =  19.802287764726785 %

Average dip for the trade =  -6.705891413670304 %
 Maximum dip for the trade =  -19.802287764726785 %
Sharpe_like Ratio =  -0.2560174861226964


# **PART 2**

In [108]:
vix= pd.read_csv("/content/hist_india_vix_-22-05-2024-to-22-05-2025.csv")
vix=vix.rename(columns={'Close ': 'Close',"Date ":"Date"})
vix

,Date,Open,High,Low,Close,Prev. Close,Change,% Change
0,22-MAY-2024,21.8100,22.2925,18.1850,21.4675,21.8100,-0.34,-1.57
1,23-MAY-2024,21.4675,24.2150,18.8400,21.3800,21.4675,-0.09,-0.41
2,24-MAY-2024,21.3800,21.8975,20.5900,21.7100,21.3800,0.33,1.54
3,27-MAY-2024,21.7100,26.1950,18.4350,23.1925,21.7100,1.48,6.83
4,28-MAY-2024,23.1925,24.4750,22.0250,24.1950,23.1925,1.00,4.32
...,...,...,...,...,...,...,...,...
243,15-MAY-2025,17.2250,17.6725,16.7200,16.8900,17.2250,-0.34,-1.94
244,16-MAY-2025,16.8925,17.0550,16.2150,16.5500,16.8925,-0.34,-2.03
245,19-MAY-2025,16.5500,17.4475,15.8275,17.3600,16.5500,0.81,4.89
246,20-MAY-2025,17.3550,17.6600,15.9675,17.3900,17.3550,0.04,0.20


**Finding mean, std. and thresholds**

In [109]:
mean=vix["Close"].mean()
std=vix["Close"].std()
print("mean:",mean, " std:",std)
Threshold1= mean + 0.5*std
Threshold2= mean + 1.5*std

mean: 15.119274193548387  std: 2.5080443809512634


**Merging ticker data with vix**

In [110]:
def merge(new_data):
  vix['Date'] = pd.to_datetime(vix['Date'])
  new_data = new_data.merge(vix[['Date', 'Close']].rename(columns={'Close': 'Vix'}), on='Date', how='left')
  return new_data
merge(bel)

<ipython-input-110-9c1324d6657d>:2: UserWarning:

Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.



,Date,Close,High,Low,Open,Volume,signal1,signal2,signal3,Capital,Vix
0,2024-05-22,281.276062,282.168681,262.134197,274.730139,85801218,0,0,0,100000.000000,21.4675
1,2024-05-23,286.135895,290.896549,280.879338,283.953911,70631009,0,0,0,100000.000000,21.3800
2,2024-05-24,294.764587,297.442444,283.656354,285.639965,63609206,0,0,0,100000.000000,21.7100
3,2024-05-27,292.037170,300.715469,287.276485,299.277339,49890427,0,0,0,100000.000000,23.1925
4,2024-05-28,286.730988,293.326488,282.714157,293.276910,33893805,0,0,0,100000.000000,24.1950
...,...,...,...,...,...,...,...,...,...,...,...
244,2025-05-16,363.899994,371.149994,349.600006,351.600006,68855818,0,0,0,91720.672119,16.5500
245,2025-05-19,363.750000,373.500000,362.549988,372.899994,52003735,0,0,0,95244.172119,17.3600
246,2025-05-20,363.799988,371.000000,358.500000,369.399994,67088831,0,0,0,95205.023712,17.3900
247,2025-05-21,383.000000,383.899994,364.000000,366.899994,88688170,0,0,0,95218.070526,17.5500


**Allocating weight based on threshold**

In [111]:
def alloc(new_data):
  i=0
  while i<len(vix):
    if vix["Close"][i]<Threshold1:
      new_data.loc[i,"allocate"]=1
    elif vix["Close"][i]>=Threshold1 and vix["Close"][i]<Threshold2:
      new_data.loc[i,"allocate"]=0.75
    elif vix["Close"][i]>=Threshold2:
      new_data.loc[i,"allocate"]=0.5
    i+=1
  return new_data
alloc(bel)

Price,Date,Close,High,Low,Open,Volume,signal1,signal2,signal3,Capital,allocate
0,2024-05-22,281.276062,282.168681,262.134197,274.730139,85801218,0,0,0,100000.000000,0.50
1,2024-05-23,286.135895,290.896549,280.879338,283.953911,70631009,0,0,0,100000.000000,0.50
2,2024-05-24,294.764587,297.442444,283.656354,285.639965,63609206,0,0,0,100000.000000,0.50
3,2024-05-27,292.037170,300.715469,287.276485,299.277339,49890427,0,0,0,100000.000000,0.50
4,2024-05-28,286.730988,293.326488,282.714157,293.276910,33893805,0,0,0,100000.000000,0.50
...,...,...,...,...,...,...,...,...,...,...,...
244,2025-05-16,363.899994,371.149994,349.600006,351.600006,68855818,0,0,0,91720.672119,0.75
245,2025-05-19,363.750000,373.500000,362.549988,372.899994,52003735,0,0,0,95244.172119,0.75
246,2025-05-20,363.799988,371.000000,358.500000,369.399994,67088831,0,0,0,95205.023712,0.75
247,2025-05-21,383.000000,383.899994,364.000000,366.899994,88688170,0,0,0,95218.070526,0.75


**Forming a trade_index and calculating new portfolio value**

In [112]:
def vix_capital(new_data,signal):
    trade_index= []
    i=0
    while i<len(new_data):
       if new_data[signal].iloc[i]== "buy" or new_data[signal].iloc[i]== "sell":
         trade_index.append(i)
       i+=1
    cap=[100000]
    j=0
    amt=100000
    quantity=0
    i=0
    while i<len(new_data)-1:
        if j< len(trade_index) and i==trade_index[j]:
            if new_data[signal].iloc[i]== "buy":
                quantity = int(amt*new_data.loc[i,"allocate"] / new_data['Close'].iloc[i])
                amt -= quantity*new_data['Close'].iloc[i]
            elif new_data[signal].iloc[i]== "sell":
                amt += quantity*new_data['Close'].iloc[i]
                quantity=0
            cap.append(amt)
            j+=1
        else:
            cap.append(amt + quantity*new_data['Close'].iloc[i])
        i+=1
    new_data['new_capital']=cap
    return new_data
vix_capital(bel,"signal2")

Price,Date,Close,High,Low,Open,Volume,signal1,signal2,signal3,Capital,allocate,new_capital
0,2024-05-22,281.276062,282.168681,262.134197,274.730139,85801218,0,0,0,100000.000000,0.50,100000.000000
1,2024-05-23,286.135895,290.896549,280.879338,283.953911,70631009,0,0,0,100000.000000,0.50,100000.000000
2,2024-05-24,294.764587,297.442444,283.656354,285.639965,63609206,0,0,0,100000.000000,0.50,100000.000000
3,2024-05-27,292.037170,300.715469,287.276485,299.277339,49890427,0,0,0,100000.000000,0.50,100000.000000
4,2024-05-28,286.730988,293.326488,282.714157,293.276910,33893805,0,0,0,100000.000000,0.50,100000.000000
...,...,...,...,...,...,...,...,...,...,...,...,...
244,2025-05-16,363.899994,371.149994,349.600006,351.600006,68855818,0,0,0,91720.672119,0.75,93632.239258
245,2025-05-19,363.750000,373.500000,362.549988,372.899994,52003735,0,0,0,95244.172119,0.75,97236.739258
246,2025-05-20,363.799988,371.000000,358.500000,369.399994,67088831,0,0,0,95205.023712,0.75,97196.690887
247,2025-05-21,383.000000,383.899994,364.000000,366.899994,88688170,0,0,0,95218.070526,0.75,97210.037628


In [113]:
fig=go.Figure()
fig.add_trace(go.Scatter(x=vix["Date"], y=vix_capital(bel,"signal2")["new_capital"], line = dict(color='blue', width=1)))
fig.update_layout(title=dict(text="New_Portfolio value over time",x=0.45,y=0.9,font=dict(size=30)),yaxis_title="portfolio value",xaxis_title="Date")
fig.show()

**Metrics(VaR,CVaR,Sharpe,Net_return)**

In [114]:
def metrics(new_data):
  new_data['dailyreturn'] = new_data['new_capital'].pct_change().fillna(0)
  confidence = 0.95
  net_return= new_data['new_capital'].iloc[-1]-100000
  var= np.percentile(new_data['dailyreturn'], (1-confidence) * 100)
  cvar= new_data[new_data['dailyreturn'] <= var]['dailyreturn'].mean()
  sharpe= new_data['dailyreturn'].mean()/new_data['dailyreturn'].std()
  print("Net_return=",net_return)
  print("VaR=",var)
  print("CVaR=",cvar)
  print("Sharpe_like=",sharpe)
metrics(bel)

Net_return= 2336.440887451172
VaR= -0.03023693002567881
CVaR= -0.5141295767036673
Sharpe_like= 0.13818343483560258
